# Analyze Control File Duplicates

**Purpose:** Analyze duplicate rows in control/auxiliary files to understand data quality issues.

**Analysis:**
- Count total rows vs unique keys
- Distribution of duplicate counts
- Check if duplicates have consistent values (safe to drop) or inconsistent (data issue)

In [ ]:
# Parameters (can be overridden by papermill)
data_path = "/Users/Mach/dev/aps/data/2026_Dmodel_data"
control_file = "aux_dataset_car_vindate_ctl.parquet"
join_key = "vin_date"
value_columns = ["pred_pp_coll", "pred_pp_liab_3p"]  # Columns to check consistency

In [ ]:
import pandas as pd
import sys
from pathlib import Path

# Add lib to path
sys.path.insert(0, str(Path.cwd().parent / 'lib'))

from data_quality import (
    analyze_duplicates,
    check_duplicate_consistency,
    get_sample_duplicates,
    summarize_duplicates
)

print('Data Quality Utilities Loaded')

In [ ]:
# Load control file
file_path = f"{data_path}/{control_file}"
print(f"Loading: {file_path}")

# Determine columns to load
cols_to_load = [join_key] + [col for col in value_columns if col]

df = pd.read_parquet(file_path, columns=cols_to_load)
print(f"\nLoaded: {len(df):,} rows x {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Analyze duplicates
print("\n" + "="*60)
print("DUPLICATE ANALYSIS")
print("="*60)

results = analyze_duplicates(df, join_key, value_columns)

print(f"\nTotal Rows: {results['total_rows']:,}")
print(f"Unique Keys: {results['unique_keys']:,}")
print(f"Duplicate Ratio: {results['duplicate_ratio']:.2f}x")
print(f"Keys with Duplicates: {results['num_duplicated_keys']:,}")
print(f"Max Duplicates per Key: {results['max_duplicates']}")
print(f"Avg Duplicates per Key: {results['avg_duplicates']:.2f}")

In [ ]:
# Show duplicate distribution
print("\n" + "="*60)
print("DUPLICATE DISTRIBUTION (Top 20)")
print("="*60)
print(f"{'Dup Count':<12} {'# Keys':<15} {'% of Total Keys'}")
print("-"*45)

dup_dist = results['duplicate_distribution']
total_keys = results['unique_keys']

for count, num_keys in sorted(dup_dist.items())[:20]:
    pct = (num_keys / total_keys) * 100
    print(f"{count:<12} {num_keys:<15,} {pct:>6.2f}%")

In [ ]:
# Check value consistency
if 'value_consistency' in results:
    print("\n" + "="*60)
    print("VALUE CONSISTENCY CHECK")
    print("="*60)
    
    for col, consistency in results['value_consistency'].items():
        print(f"\nColumn: {col}")
        print("-"*40)
        
        if 'error' in consistency:
            print(f"  ERROR: {consistency['error']}")
        else:
            status = "✓ CONSISTENT" if consistency['consistent'] else "✗ INCONSISTENT"
            print(f"  Status: {status}")
            print(f"  Consistency: {consistency['pct_consistent']:.1f}%")
            print(f"  Duplicated Keys: {consistency['total_duplicated_keys']:,}")
            print(f"  Inconsistent Keys: {consistency['inconsistent_keys']:,}")
else:
    print("\nNo value consistency check performed")

In [ ]:
# Show sample duplicates
print("\n" + "="*60)
print("SAMPLE DUPLICATE ROWS (First 3 Keys)")
print("="*60)

sample_df = get_sample_duplicates(df, join_key, n_samples=3)
print(f"\nShowing {len(sample_df)} rows from {sample_df[join_key].nunique()} duplicate keys:\n")
print(sample_df.to_string(index=False))

In [ ]:
# Summary and recommendation
print("\n" + "="*60)
print("SUMMARY & RECOMMENDATION")
print("="*60)

print(f"\n1. File: {control_file}")
print(f"2. Join Key: {join_key}")
print(f"3. Duplicate Ratio: {results['duplicate_ratio']:.2f}x (ideal: 1.0x)")

if results['duplicate_ratio'] > 1.1:
    print("\n⚠️  HIGH DUPLICATE RATIO DETECTED")
    
    # Check if safe to drop
    all_consistent = True
    if 'value_consistency' in results:
        for col, consistency in results['value_consistency'].items():
            if not consistency.get('consistent', False):
                all_consistent = False
                break
    
    if all_consistent:
        print("\n✓ SAFE TO DROP DUPLICATES")
        print("  All duplicate keys have identical values.")
        print("  Recommendation: Use drop_duplicates(subset='vin_date') before merge.")
    else:
        print("\n✗ NOT SAFE TO DROP")
        print("  Some duplicate keys have different values - data quality issue!")
        print("  Recommendation: Investigate root cause before proceeding.")
else:
    print("\n✓ No significant duplication issues")

print("\nExpected row count after deduplication: {:,}".format(results['unique_keys']))